In [3]:
import pandas as pd
import numpy as np
import datetime as dt
import sqlite3
from sqlalchemy import create_engine


In [6]:
engine = create_engine("sqlite:///C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/database/managing.db")

In [ ]:
# read csv customer file
customers = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/customers.csv")
customers.head()

In [8]:
# load to sql
customers.to_sql("customers", engine, if_exists='replace', index=False)

10040

In [ ]:
pd.read_sql_query('Select * from customers', engine)

In [ ]:
# # python practice
# isnull()
# fillna()
# drop_duplicates()
# .duplicated()
# type conversion
# outlier detection
# invalid value detection
# string cleaning

# gender and age have null value


In [24]:
customers = customers.drop_duplicates(subset='customer_id').reset_index(drop=True)

In [ ]:
customers.info()

In [ ]:
customers.info()

In [28]:
# for formating the date column and create new columns for it
customers['join_date'] = pd.to_datetime(customers['join_date'], format='mixed', errors='coerce')
customers['join_month'] = customers['join_date'].dt.to_period('M').astype(str)
customers['join_year'] = customers['join_date'].dt.year


In [ ]:
customers['join_date'].value_counts()

In [ ]:
customers['join_date'].value_counts()

In [38]:
# remove na in gender
customers = customers[customers['gender'].isna() == False]

In [ ]:
customers.info()

In [48]:
customers['age'] = customers['age'].fillna(customers.groupby(['gender', 'state'])['age'].transform('median'))
# customers.groupby(['gender', 'state'])['age'].transform('median')

In [ ]:
# age outliers
min_age = customers['age'].quantile(0.25)
max_age = customers['age'].quantile(0.75)

iqr = max_age - min_age

lower_age = min_age - 1.5 * iqr
upper_age = max_age + 1.5 * iqr

outlier = customers[
    (
        customers['age'] < lower_age
    ) | (
        customers['age'] > upper_age
    )
]

customers = customers[
    (
        customers['age'] >= lower_age
    ) & (
        customers['age'] <= upper_age
    )
]

print(f"Number of outliers: {len(outlier)}")
print(f"Percentage: {len(outlier) / len(customers) * 100:.2f}%")

In [ ]:
customers['age'].min()

In [53]:
# add the customers data load into sql database
customers.to_sql("customers", engine, if_exists='replace' , index=False )

9785

In [ ]:
pd.read_sql_query('select * from customers limit 5', engine)

In [55]:
# # sql questions 
# Find duplicate customers.
# Find customers with invalid ages.
# Find future join dates.
# Find missing genders.
# Count customers by segment.
# Top states by customer count.
# Monthly customer acquisition.

In [ ]:
# Find duplicate customers.
pd.read_sql_query("""
with dupp_value as (
    select customer_id, row_number() over(partition by customer_id order by join_date) as rnk
    from customers)
    select customer_id
    from dupp_value
    where rnk > 1
""" , engine)

In [ ]:
# Find customers with invalid ages.
pd.read_sql_query('''
    select 
        * from customers
    where age is not null and age > 0 and age < 120
''', engine)

In [ ]:
# Find future join dates.
pd.read_sql_query('''
select * from customers where join_date > current_date ''', engine)

In [ ]:
# Find missing genders.
pd.read_sql_query("""
select customer_id, customer_name, gender  from customers where gender is null """, engine)

In [ ]:
# Count customers by segment.
pd.read_sql_query('''
select customer_segment , count(customer_id) as total_customer_count from customers group by customer_segment order by total_customer_count desc''', engine)

In [ ]:
# Top states by customer count.
pd.read_sql_query('''
select state , count(customer_id) as total_customer_count from customers group by state order by total_customer_count desc''', engine)

In [ ]:
# Monthly customer acquisition.
pd.read_sql_query('''
select join_month as monthly_joined , count(customer_id) as customers_acquire from customers group by join_month order by join_month asc''', engine)